# Political Risk Events, Oil Market Volatility, and Investment Correlations During Middle East Conflict Periods: Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The workflow follows the Croissant schema and makes extensive use of `@id` references for record sets and fields, ensuring precise and reproducible data handling.

### Dataset Source
The dataset metadata is defined by a Croissant schema, available at:

**https://sen.science/doi/10.71728/senscience.19ns-r3ze/fair2.json**


In [ ]:
# Install mlcroissant (if not already installed)
!pip install -U mlcroissant

## 1. Data Loading

We load the dataset metadata and inspect the high-level summary using `mlcroissant`:

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.19ns-r3ze/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level info
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description[:200]}...")  # Preview first 200 chars
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview

Let's view available record sets and their fields using the Croissant `@id`. This step helps us identify which tables and columns are present for further exploration. For full reproducibility, every entity is referenced by its `@id`.


In [ ]:
# List record sets and their fields, always by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    fields = rs.get('field', [])
    # 'field' can be dict or list of dicts
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields / Columns:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
    print()

## 3. Data Extraction

Now we extract records for each record set of interest into pandas DataFrames using the `@id` for record sets and fields.

> **Note:** If the dataset is large or has sensitive data, you may want to limit reading with e.g. `islice`.


In [ ]:
from collections import OrderedDict

# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {df.shape[0]} rows, columns: {list(df.columns)}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")
    print()
# Preview columns for the first available data table
if dataframes:
    first_rsid = list(dataframes.keys())[0]
    print(f"Columns in {first_rsid}:")
    print(dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())

## 4. Exploratory Data Analysis (EDA)

We'll select a numerical field from one of the available record sets and apply basic exploration: filtering by a value threshold, normalization, and optional group statistics. All fields are referenced strictly by their `@id` when indexing.

> **Edit below** to fit the record set and field IDs from the previous overview cell for your analysis. Example uses are generic and may need to be customized per actual column names.

In [ ]:
# EDIT: Set the record set @id and numeric field @id you want to analyze
# Example: (Update these based on your previous output above)
record_set_id = list(dataframes.keys())[0] if dataframes else None
if record_set_id is not None:
    df = dataframes[record_set_id]
    # Try to find a likely numeric column
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # use first available
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Optionally group by a categorical field
        # Try to find a non-numeric column for grouping
        group_field_candidates = [col for col in df.columns if df[col].dtype == object]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped means by {group_field}:")
            display(grouped.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No available record set for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and examine relationships with a categorical field if available. Visualization uses pandas/Matplotlib. (You may further enhance these visualizations depending on dataset content.)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and numeric_fields:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if categorical exists
    if group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xticks(rotation=30, ha='right')
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to use `mlcroissant` to load, inspect, process, and visualize data from a FAIR-compliant Croissant dataset. By referencing all entities (record sets, fields) via their `@id`, analysis remains robust and reproducible even as schemas evolve. 

For further data science or machine learning tasks, continue with domain-specific feature engineering, modeling, or cross-dataset correlation as needed.
